## 1. Setup and import

In [56]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper
from topologicpy.Grid import Grid
from topologicpy.Graph import Graph
from topologicpy.Color import Color


from utils import reset_dictionaries, transfer_dicts_by_key

In [57]:
renderer = "vscode"

In [ ]:
floorplan = Topology.ByBREPPath(r"Assets\Gleiss21\Gleiss21_GF_face.brep")

print(floorplan)

Topology.ByBREPPath - Error: the BREP file is not a valid file. Returning None.
None


In [90]:
Topology.Show(floorplan,
              camera=[0,0,6],
              faceColor=[210,210,250],
              faceOpacity=1,
              edgeColor="white",
              edgeWidth=3,
              showVertices=False,
              backgroundColor="black",
              width=800,
              height=600,
              renderer = renderer)

Topology.Show - Error: the input topologies parameter does not contain any valid topology. Returning None.


## 2. Create graph vertices and isovist vertices

In [76]:
b_r = Wire.BoundingRectangle(floorplan)
d = Topology.Dictionary(b_r)
xmin = Dictionary.ValueAtKey(d, "xmin")
xmax = Dictionary.ValueAtKey(d, "xmax")
ymin = Dictionary.ValueAtKey(d, "ymin")
ymax = Dictionary.ValueAtKey(d, "ymax")
width = Dictionary.ValueAtKey(d, "width")
length = Dictionary.ValueAtKey(d, "length")

In [77]:
# Isovists grid
uRange1 = list(range(0,int(width)+9,9))
vRange1 = list(range(0,int(length)+9,9))

grid1 = Grid.VerticesByDistances(floorplan, clip=True, uRange=uRange1, vRange=vRange1)

iso_verts = Topology.Vertices(grid1)

In [78]:
# Analysis graph grid
uRange2 = list(range(0,int(width)+3,3))
vRange2 = list(range(0,int(length)+3,3))

grid2 = Grid.EdgesByDistances(floorplan, clip=True, uRange=uRange2, vRange=vRange2)

In [79]:
shell = Topology.Slice(floorplan, grid2)
faces = Topology.Faces(shell)
# Assign a sequential unique face id to reference it later (e.g. "face_21")
for i, f in enumerate(faces):
    d = Dictionary.ByKeyValue("face_id", "face_"+str(i+1))
    f = Topology.SetDictionary(f, d)

In [80]:
# Note: Graph nodes automatically inherit the dictionaries of the entities they 
analysis_graph = Graph.ByTopology(shell)

In [81]:
g_verts = Graph.Vertices(analysis_graph)
print("Number of graph vertices:", len(g_verts))

Number of graph vertices: 1940


In [82]:
Topology.Show(analysis_graph,
              camera=[0,0,6],
              vertexSize=4,
              vertexColor="red",
              edgeColor="lightgrey",
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

## 3. Visibility Graph Analysis

In [83]:
isovists = []
for v in iso_verts:
    isovist = Face.Isovist(floorplan, v)
    isovists.append(isovist)

Face.Isovist - Error: Could not convert visible polygon to a TopologicPy Face. Returning None.


KeyboardInterrupt: 

In [68]:
Topology.Show(floorplan, isovists,
              faceColorKey="cp_color",
              faceOpacity=0.3,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="black",
              width=800,
              height=600,
              renderer=renderer)

In [84]:
new_verts = []
n_list = []
for i, iso in enumerate(isovists):
    if iso: # Skip is iso is None
        v = iso_verts[i]
        b_list = Vertex.IsInternal2D(g_verts, iso)
        b_list = [b for b in b_list if b]
        n = len(b_list)
        n_list.append(n)
        d = Dictionary.ByKeyValue("visibility", n)
        v = Topology.SetDictionary(v, d)
        new_verts.append(v)

In [85]:
for v in g_verts:
    new_v = Vertex.InterpolateValue(v, vertices=new_verts, n=2, key="visibility")

In [86]:
minValue = min(n_list)
maxValue = max(n_list)
for v in g_verts:
    d = Topology.Dictionary(v)
    vb = Dictionary.ValueAtKey(d, "visibility")
    color = Color.AnyToHex(Color.ByValueInRange(vb, minValue=minValue, maxValue=maxValue, colorScale="thermal"))
    d = Dictionary.SetValueAtKey(d, "vb_color", color)
    d = Dictionary.SetValueAtKey(d, "size", 16)
    v = Topology.SetDictionary(v, d)

In [87]:
reset_dictionaries(shell)
_ = transfer_dicts_by_key(faces, g_verts, "face_id")

In [88]:
Topology.Show(faces,
              faceColorKey="vb_color",
              faceOpacity=1,
              showEdges=False,
              showVertices=False,
              camera=[0,0,6],
              backgroundColor="white",
              width=800,
              height=600,
              renderer=renderer)